# Query 1: Filtering with Complex Conditions
**Type:** Filtering  
**Problem Statement:** Find all trips where fare > $20 AND passengers > 1. Average fare in this dataset is ~$12, so $20 is a meaningful threshold that filters to longer/premium trips.

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

spark = SparkSession.builder \
    .appName('Q1_Filtering') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.driver.memory', '2g') \
    .config('spark.executor.memory', '2g') \
    .config('spark.eventLog.enabled', 'false') \
    .config('spark.ui.enabled', 'false') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

26/04/25 19:37:38 WARN Utils: Your hostname, mariam-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/04/25 19:37:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/25 19:37:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1


----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 44542)
Traceback (most recent call last):
  File "/usr/lib/python3.10/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.10/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.10/socketserver.py", line 747, in __init__
    self.handle()
  File "/home/mariam/.local/lib/python3.10/site-packages/pyspark/accumulators.py", line 295, in handle
    poll(accum_updates)
  File "/home/mariam/.local/lib/python3.10/site-packages/pyspark/accumulators.py", line 267, in poll
    if self.rfile in r and func():
  File "/home/mariam/.local/lib/python3.10/site-packages/pyspark/accumulators.py", line 271, in accum_updates


In [2]:
# Load data with 20% sample to avoid disk/memory issues
df = spark.read.option('header', 'true').option('inferSchema', 'true') \
          .csv('../data/yellow_tripdata_2015-01.csv') \
          .sample(fraction=0.2, seed=42)

# Rename columns to match project convention
df = df.withColumnRenamed('tpep_pickup_datetime',  'pickup_datetime') \
       .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime') \
       .withColumnRenamed('fare_amount',            'fare') \
       .withColumnRenamed('passenger_count',        'passengers') \
       .withColumnRenamed('trip_distance',          'distance') \
       .withColumnRenamed('total_amount',           'total')

# Cast correct types
df = df.withColumn('fare',       F.col('fare').cast(DoubleType())) \
       .withColumn('total',      F.col('total').cast(DoubleType())) \
       .withColumn('distance',   F.col('distance').cast(DoubleType())) \
       .withColumn('passengers', F.col('passengers').cast(IntegerType())) \
       .withColumn('tip_amount', F.col('tip_amount').cast(DoubleType())) \
       .cache()

df.createOrReplaceTempView('trips')
rdd = df.rdd
print('Total rows (20% sample):', df.count())
print('Columns:', df.columns)

Total rows (20% sample): 2233826
Columns: ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passengers', 'distance', 'pickup_longitude', 'pickup_latitude', 'RateCodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total']


## RDD Implementation

In [3]:
start = time.time()

# No optimizer. Full scan with Python lambda filter.
result_rdd = (
    rdd
    .filter(lambda r: r['fare']       is not None
                  and r['passengers'] is not None
                  and r['fare'] > 20
                  and r['passengers'] > 1)
    .count()
)

rdd_time = time.time() - start
print(f'RDD result: {result_rdd:,} trips | Time: {rdd_time:.2f}s')

RDD result: 79,679 trips | Time: 11.88s


## DataFrame Implementation

In [4]:
start = time.time()

# Catalyst applies predicate pushdown – filter evaluated before full scan.
result_df = df.filter((F.col('fare') > 20) & (F.col('passengers') > 1))

print('--- Q1 DataFrame explain(True) ---')
result_df.explain(True)
result_df.show(10)

df_count = result_df.count()
df_time = time.time() - start
print(f'DataFrame result: {df_count:,} trips | Time: {df_time:.2f}s')

--- Q1 DataFrame explain(True) ---
== Parsed Logical Plan ==
'Filter (('fare > 20) AND ('passengers > 1))
+- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, cast(tip_amount#32 as double) AS tip_amount#256, tolls_amount#33, improvement_surcharge#34, total#196]
   +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, cast(passengers#116 as int) AS passengers#236, distance#216, pickup_longitude#22, pickup_latitude#23, RateCodeID#24, store_and_fwd_flag#25, dropoff_longitude#26, dropoff_latitude#27, payment_type#28, fare#176, extra#30, mta_tax#31, tip_amount#32, tolls_amount#33, improvement_surcharge#34, total#196]
      +- Project [VendorID#17, pickup_datetime#55, dropoff_datetime#76, passengers#116, cast(distance#136 as double) AS distance#216, pickup_longitude#22,

+--------+-------------------+-------------------+----------+--------+-------------------+------------------+----------+------------------+------------------+------------------+------------+----+-----+-------+----------+------------+---------------------+-----+
|VendorID|    pickup_datetime|   dropoff_datetime|passengers|distance|   pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total|
+--------+-------------------+-------------------+----------+--------+-------------------+------------------+----------+------------------+------------------+------------------+------------+----+-----+-------+----------+------------+---------------------+-----+
|       1|2015-01-03 08:58:44|2015-01-03 09:19:31|         2|    10.8|-73.989883422851562|40.746803283691406|       1.0|                 Y|-73.86493682861328|40.770469665527344|         2.0|31.0|  0.0|    0.5|     

## Spark SQL Implementation

In [5]:
start = time.time()

result_sql = spark.sql("""
    SELECT   VendorID, pickup_datetime, fare, passengers,
             distance, payment_type
    FROM     trips
    WHERE    fare > 20
      AND    passengers > 1
    ORDER BY fare DESC
""")

print('--- Q1 Spark SQL explain(True) ---')
result_sql.explain(True)
result_sql.cache()
sql_count = result_sql.count()
sql_time = time.time() - start
print(f'SQL result: {sql_count:,} trips | Time: {sql_time:.2f}s')
result_sql.show(10)
result_sql.unpersist()

--- Q1 Spark SQL explain(True) ---
== Parsed Logical Plan ==
'Sort ['fare DESC NULLS LAST], true
+- 'Project ['VendorID, 'pickup_datetime, 'fare, 'passengers, 'distance, 'payment_type]
   +- 'Filter (('fare > 20) AND ('passengers > 1))
      +- 'UnresolvedRelation [trips], [], false

== Analyzed Logical Plan ==
VendorID: int, pickup_datetime: timestamp, fare: double, passengers: int, distance: double, payment_type: double
Sort [fare#176 DESC NULLS LAST], true
+- Project [VendorID#17, pickup_datetime#55, fare#176, passengers#236, distance#216, payment_type#28]
   +- Filter ((fare#176 > cast(20 as double)) AND (passengers#236 > 1))
      +- SubqueryAlias trips
         +- View (`trips`, [VendorID#17,pickup_datetime#55,dropoff_datetime#76,passengers#236,distance#216,pickup_longitude#22,pickup_latitude#23,RateCodeID#24,store_and_fwd_flag#25,dropoff_longitude#26,dropoff_latitude#27,payment_type#28,fare#176,extra#30,mta_tax#31,tip_amount#256,tolls_amount#33,improvement_surcharge#34,total#196

SQL result: 79,679 trips | Time: 2.99s
+--------+-------------------+------+----------+--------+------------+
|VendorID|    pickup_datetime|  fare|passengers|distance|payment_type|
+--------+-------------------+------+----------+--------+------------+
|       1|2015-01-24 12:43:36| 780.0|         4|     0.0|         1.0|
|       1|2015-01-19 14:51:05|750.01|         2|     0.2|         4.0|
|       2|2015-01-27 10:15:47| 499.0|         2|     0.0|         1.0|
|       1|2015-01-14 12:01:10| 450.0|         2|    99.9|         1.0|
|       1|2015-01-04 05:40:11| 405.0|         3|     8.7|         2.0|
|       1|2015-01-12 05:58:12| 347.0|         2|    99.9|         1.0|
|       1|2015-01-03 19:35:52| 325.0|         3|     0.0|         1.0|
|       2|2015-01-09 23:03:49| 310.0|         2|    0.22|         1.0|
|       1|2015-01-02 10:22:05| 279.7|         2|    73.1|         1.0|
|       1|2015-01-07 18:12:28| 277.0|         3|     0.0|         1.0|
+--------+-------------------+------+-

DataFrame[VendorID: int, pickup_datetime: timestamp, fare: double, passengers: int, distance: double, payment_type: double]

## Performance Comparison

In [6]:
print('='*65)
row1 = f'{"Metric":<25} {"RDD":>12} {"DataFrame":>12} {"SQL":>12}'
row2 = f'{"Execution Time":<25} {rdd_time:>11.2f}s {df_time:>11.2f}s {sql_time:>11.2f}s'
row3 = f'{"Result Count":<25} {result_rdd:>12,} {df_count:>12,} {sql_count:>12,}'
row4 = f'{"Predicate Pushdown":<25} {"No":>12} {"Yes":>12} {"Yes":>12}'
row5 = f'{"Optimizer":<25} {"None":>12} {"Catalyst":>12} {"Catalyst":>12}'
print(row1)
print('-'*65)
print(row2)
print(row3)
print(row4)
print(row5)
print('='*65)
print()
print('KEY INSIGHT:')
print('RDD performs full scan with Python lambdas — no optimization.')
print('DataFrame/SQL use Catalyst predicate pushdown to filter early.')
print('Both structured APIs are significantly faster than RDD.')

Metric                             RDD    DataFrame          SQL
-----------------------------------------------------------------
Execution Time                  11.88s        1.45s        2.99s
Result Count                    79,679       79,679       79,679
Predicate Pushdown                  No          Yes          Yes
Optimizer                         None     Catalyst     Catalyst

KEY INSIGHT:
RDD performs full scan with Python lambdas — no optimization.
DataFrame/SQL use Catalyst predicate pushdown to filter early.
Both structured APIs are significantly faster than RDD.
